# The Graves That Answer — Episode 1 / Colab recovery

Uses your existing **T4 runtime**. Run all cells once. This notebook reads the approved private D05 and storyboard from your own Google Drive, verifies all **42 chapters** and the exact two-chapter Episode 1 text, then reuses the previously implemented Qwen Ryan + manhua workflow. It generates **only missing assets**, saves checkpoints to your private Drive and assembles a **1920×1080 review MP4** on CPU.

Your manuscript and assets are never committed to GitHub. Google Drive access requires one authorization. The final MP4 is **not a publication approval**: listen through the voice and inspect images before uploading to YouTube.

**Note:** Free Colab can disconnect or run out of GPU availability. If that happens, reconnect and rerun: verified checkpoints in `MyDrive/Audiobook_Studio/The_Graves_That_Answer_Ep01/progress` are reused. Do not switch to another voice without approving its audition.

In [ ]:
# Existing T4 runtime: run this notebook top to bottom. Private story stays in YOUR Drive/Colab.
import os, sys, re, json, pathlib, hashlib, subprocess, shutil, urllib.request
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No Colab GPU attached. Reconnect to the existing T4 runtime; no CPU model download attempted.")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qwen-tts", "accelerate", "soundfile", "diffusers", "transformers", "safetensors", "pillow", "google-api-python-client"], check=True)
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
service = build("drive", "v3", credentials=credentials, cache_discovery=False)
def private_doc_exact(title):
    quoted = title.replace("\\", "\\\\").replace("'", "\\'")
    q = "name = '" + quoted + "' and mimeType = 'application/vnd.google-apps.document' and trashed = false"
    records = service.files().list(q=q, fields="nextPageToken,files(id,name)", page_size=100).execute().get("files", [])
    if len(records) != 1:
        raise RuntimeError("Expected exactly one approved private Google Doc titled: " + title)
    raw = service.files().export(fileId=records[0]["id"], mimeType="text/plain").execute()
    return raw.decode("utf-8-sig").replace("\r\n", "\n").replace("\r", "\n")
source = private_doc_exact("The Graves That Answer — Complete D05 Manuscript — Three Levels — Five Volumes 42 Chapters")
approved = private_doc_exact("The Graves That Answer — Episode 01 — Approved D05 Narration Source")
board = private_doc_exact("The Graves That Answer — Episode 01 — Storyboard, Cast, Narration & Production Checklist")
chapter_regex = re.compile(r"(?m)^CHAPTER [A-Z]+(?:[- ][A-Z]+)*\s*$")
hits = list(chapter_regex.finditer(source))
if len(hits) != 42:
    raise RuntimeError("D05 canonical manuscript must have 42 chapters; stop without inventing content.")
chapters = [source[m.start():(hits[i+1].start() if i+1 < len(hits) else len(source))].strip() for i,m in enumerate(hits)]
norm = lambda s: " ".join(s.split())
ep1 = "\n\n".join(chapters[:2])
approved_body = approved[approved.index("CHAPTER ONE"):].strip()
if norm(ep1) != norm(approved_body):
    raise RuntimeError("Approved Episode 1 differs from D05 master. No voice generation.")
root = pathlib.Path("/content/graves_private")
root.mkdir(parents=True, exist_ok=True)
parts = {}
for i in range(0, 42, 2):
    filename = root / ("ep%03d.txt" % (i//2 + 1))
    filename.write_text("\n\n".join(chapters[i:i+2]), encoding="utf-8")
    parts[i//2 + 1] = filename
manuscript = root / "D05_chapter_text_42.txt"
manuscript.write_text("\n\n".join(f.read_text(encoding="utf-8") for f in parts.values()), encoding="utf-8")
if norm(manuscript.read_text(encoding="utf-8")) != norm(source[hits[0].start():]):
    raise RuntimeError("42-chapter source coverage mismatch")
cast_section = board.split("VISUAL IDENTITY PROPOSALS", 1)[1].split("Setting:", 1)[0]
cast = []
for line in cast_section.splitlines():
    if ":" not in line: continue
    name, description = line.split(":", 1)
    if not name.strip() or not description.strip(): continue
    key = re.sub(r"[^a-z0-9-]", "-", name.strip().lower().replace(" ", "-")).strip("-")
    if key: cast.append({"id":key, "name":name.strip(), "visual_description":description.strip()})
if len(cast) < 4: raise RuntimeError("Story character references could not be read from approved storyboard")
beats = [m.group(1) for m in re.finditer(r"(?m)^\d{2}\.\s+(.+)$", board)]
if len(beats) != 20: raise RuntimeError("Expected 20 storyboard beats; do not use guessed illustrations")
# Source excerpts always override the supporting shot ideas.
briefs = [(" / ".join(beats[i:i+2])) for i in range(0,20,2)]
repo = pathlib.Path("/content/TOOLS")
for p in ["video/__init__.py", "video/audiobook_studio/__init__.py", "video/audiobook_studio/manhua_pipeline/__init__.py"]:
    dest=repo/p;dest.parent.mkdir(parents=True,exist_ok=True);dest.touch()
commit="cfc69614d3b2c14418df12bea8d2522701efb4bb"
files=["video/audiobook_studio/episode_fastlane.py",
       "video/audiobook_studio/manhua_pipeline/core.py",
       "video/audiobook_studio/manhua_pipeline/visual_quality.py",
       "video/audiobook_studio/manhua_pipeline/kaggle_runner.py",
       "video/audiobook_studio/manhua_pipeline/assemble_existing.py"]
for path in files:
    dest=repo/path
    if not dest.exists():
        url="https://raw.githubusercontent.com/eghosa001/TOOLS/"+commit+"/"+path
        urllib.request.urlretrieve(url,str(dest))
sys.path.insert(0,str(repo))
from video.audiobook_studio.manhua_pipeline.core import create_package
package=root/"MANHUA_STORY_PACKAGE.zip"
create_package(story_id="the-graves-that-answer",title="The Graves That Answer",
    manuscript=manuscript,episodes=parts,output=package,style="dark_fantasy",
    characters=cast,source_approved=True,image_model_rights_reviewed=True,
    thumbnail_hook="The Graves That Answer",
    scene_briefs={1:briefs})
private_drive=pathlib.Path("/content/drive/MyDrive/Audiobook_Studio/The_Graves_That_Answer_Ep01")
progress=private_drive/"progress"
progress.mkdir(parents=True,exist_ok=True)
os.environ.update(MANHUA_INPUT_DIR=str(root),MANHUA_PROGRESS_INPUT_DIR=str(progress),
                  MANHUA_WORK_DIR="/content/manhua-studio",MANHUA_GPU_SLOT="0")
print("SOURCE_PREFLIGHT_PASS",{"chapters":42,"approved_episode":1,"storyboard_beats":20,
       "characters":len(cast),"voice":"Qwen Ryan","gpu":torch.cuda.get_device_name(0)},flush=True)

In [ ]:
# Generate missing Ryan speech + illustrated panels, then assemble the actual private MP4.
# A checkpoint is pushed to your Drive at every validated milestone.
from video.audiobook_studio.manhua_pipeline import kaggle_runner as engine
from video.audiobook_studio.manhua_pipeline import assemble_existing
from video.audiobook_studio.episode_fastlane import sha256, validate_review_video
from argparse import Namespace
original_checkpoint = engine.checkpoint_out
def checkpoint_to_drive(meta, episode, source_dir):
    local = original_checkpoint(meta, episode, source_dir)
    target = progress/"MANHUA_PROGRESS.zip"
    staging = progress/"MANHUA_PROGRESS.partial.zip"
    shutil.copy2(local, staging)
    os.replace(staging, target)
    return local
engine.checkpoint_out = checkpoint_to_drive
result = engine.process(story_id="the-graves-that-answer",episode=1,phase="assets",quality="quality")
print("GPU_ASSET_RESULT", result,flush=True)
stage=pathlib.Path(os.environ["MANHUA_WORK_DIR"])/"the-graves-that-answer"/"ep001"
audio=stage/"narration_review.m4a"
timeline=stage/"PRIVATE_SCENE_TIMELINE.json"
thumbnail=stage/"THUMBNAIL_REVIEW.png"
qa=stage/"PRIVATE_VISUAL_QA_REPORT.json"
if not all(p.is_file() for p in [audio,timeline,thumbnail,qa]):
    raise RuntimeError("Some assets failed visual QA: successful outputs are saved in private progress. No fake MP4 generated.")
video=stage/"the_graves_that_answer_ep01_PRIVATE_REVIEW.mp4"
kwargs=dict(audio=str(audio),timeline=str(timeline),image=None,thumbnail=str(thumbnail),
            visual_qa_report=str(qa),require_visual_qa=False,
            output=str(video),preset="ultrafast",motion="none",force=False)
report=assemble_existing.assemble(Namespace(**kwargs))
destination=private_drive/video.name
temp=private_drive/(video.name+".partial")
shutil.copy2(video,temp)
os.replace(temp,destination)
if sha256(video)!=sha256(destination):
    raise RuntimeError("Drive upload checksum mismatch")
checked=validate_review_video(destination,expected_audio_seconds=report["duration_seconds"])
print("DRIVE_VIDEO_DELIVERED",str(destination),{"seconds":checked["duration_seconds"],
    "height":checked["height"],"sha256":checked["sha256"],
    "visual_review_pending":True,"publication_approved":False},flush=True)